<a href="https://colab.research.google.com/github/vahagngrigoryan2006/flyrank-internship-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vahagngrigoryan2006/flyrank-internship-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [57]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
from google.colab import userdata
from datasets import load_dataset
import pandas as pd

# Retrieve token securely from Colab Secrets
HF_TOKEN = userdata.get('HF_TOKEN')

import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
DIM_CONTENT   = f"read_parquet(\'{WAREHOUSE}/dim_content.parquet\')"
FACT_APRIL_MAY  = (
    f"read_parquet(['{WAREHOUSE}/fact_content_daily_performance/month=2026-03/*.parquet', "
    f"'{WAREHOUSE}/fact_content_daily_performance/month=2026-04/*.parquet', "
    f"'{WAREHOUSE}/fact_content_daily_performance/month=2026-05/*.parquet'])"
)

features = con.sql(f"""
    WITH bounds AS (
        SELECT DATE \'2026-05-31\' AS as_of_date
    ),
    per_item AS (
        SELECT f.client_hash_id, f.content_hash_id,
               MIN(f.report_date) AS first_seen,
               SUM(CASE WHEN f.report_date >= b.as_of_date - INTERVAL 30 DAY
                        THEN f.gsc_impressions ELSE 0 END) AS last_30_impressions,
               SUM(CASE WHEN f.report_date <  b.as_of_date - INTERVAL 30 DAY AND f.report_date >= b.as_of_date - INTERVAL 60 DAY
                        THEN f.gsc_impressions ELSE 0 END) AS prev_30_impressions,
               AVG(CASE WHEN f.report_date <  b.as_of_date - INTERVAL 30 DAY AND f.report_date >= b.as_of_date - INTERVAL 60 DAY
                        THEN f.gsc_avg_position END)       AS prev_30_avg_position
        FROM {FACT_APRIL_MAY} f, bounds b
        GROUP BY 1, 2
    )
    FROM per_item p
    JOIN {DIM_CONTENT} d USING (content_hash_id)
    WHERE p.first_seen <= DATE \'2026-05-31\' - INTERVAL 60 DAY   -- guard (a): full prev_30 history
      AND p.prev_30_impressions >= 100                             -- guard (b): activity floor
""").df()

decision_moment = pd.Timestamp("2026-05-01")
features["content_created_date"] = pd.to_datetime(features["content_created_date"])
features["content_updated_date"] = pd.to_datetime(features["content_updated_date"])
features["content_age_days_at_decision"] = (decision_moment - features["content_created_date"]).dt.days
features["days_since_last_update_at_decision"] = (decision_moment - features["content_updated_date"]).dt.days

features["keyword_created_date"] = pd.to_datetime(features["keyword_created_date"])
features["keyword_age_days_at_decision"] = (decision_moment - features["keyword_created_date"]).dt.days


features = features[features["days_since_last_update_at_decision"] > 0] # the guard (c)

print(f"Feature frame: {len(features):,} content items.")

print(features.isna().sum())



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame: 18,918 content items.
client_hash_id                            0
content_hash_id                           0
first_seen                                0
last_30_impressions                       0
prev_30_impressions                       0
prev_30_avg_position                      0
client_hash_id_1                          0
keyword_hash_id                         427
url_hash_id                               0
keyword_char_count                        0
keyword_token_count                       0
url_char_count                            0
content_created_date                      0
content_updated_date                      0
content_type                              0
search_volume                           431
competition                             431
competition_level                       527
cpc                                     431
main_intent                             526
backlinks                              6688
category_count                         

## 2. Feature notes (meaning, missing, categorical, available-when?)

All the features listed below exist before the decision day.

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

**prev_30_impressions**: The number of impressions during previous 60th day to 30th day period. No missing values

**prev_30_avg_position**: The average position of the content during previous 60th day to 30th day period. No missing values.

**content_age_days_at_decision**: days between the created day of the content and the decision day (30th day)

**days_since_last_update_at_decision**: days between the last update and the decision day (30th day) (`>0`)

**keyword_char_count**: char count of the keyword. No missing values.

**keyword_token_count**: number of tokens in the keyword.

**content_type**: Classification of the content (keyword article, feedly article, comparison article). No missing values.

**search_volume**: 	Search-volume estimate for the page's target keyword. The value is missing when there is no keyword data available. All missing values were replaced with zero, while a new boolean column **no_keyword_data** indicates the missing information.

**competition**: Keyword competition score, `0–1`. The value is missing when there is no keyword data available. All missing values were replaced with zero, while a new boolean column **no_keyword_data** indicates the missing information.

**cpc**: Cost-per-click estimate for the target keyword. The value is missing when there is no keyword data available. All missing values were replaced with zero, while a new boolean column **no_keyword_data** indicates the missing information.

**keyword_age_days_at_decision**: days between keyword entry creation day and the decision day (30th day). The value is missing when there was no keyword entry created. All missing values were replaced with `-30` to indicate there was no keyword during all 60 day period, while a new boolean column **no_keyword_data** indicates the missing information.

**main_intent**: Primary search intent of the keyword (informational, transactional, commercial, navigational). Missing values are considered a separate category.

**backlinks**: Count of external inbound links pointing to the content’s URL. All missing values were replaced with zero, while a new boolean column **backlinks_na** indicates the missing information.

**category_count**: Number of topical categories assigned to the content. No missing values.

**word_count_tier** Total word count of the content body, binned into the categories `<1000, 1000-2000, 2000-3500, 3500+, NA`.

**char_count_tier**: Total character count of the content body, binned into the categories `<8000, 8000-15000, 15000-25000, 25000+, NA`.

**no_keyword_data**: boolean, indicates whether keyword data is missing.

**backlinks_na**: boolean, indicates whether backlink information is missing.

In [65]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

selected_features = [
    "prev_30_impressions",
    "prev_30_avg_position",
    "content_age_days_at_decision",
    "days_since_last_update_at_decision",
    "keyword_char_count",
    "keyword_token_count",
    "content_type",
    "search_volume",
    "competition",
    "cpc",
    "keyword_age_days_at_decision",
    "main_intent",
    "backlinks",
    "category_count",
    "char_count",
    "word_count"
]


df = features[selected_features]

import numpy as np

bins = [-np.inf, 1000, 2000, 3500, np.inf]
labels = ["<1000", "1000-2000", "2000-3500", "3500+"]

df["word_count_tier"] = pd.cut(
    df["word_count"],
    bins=bins,
    labels=labels,
    right=False
)

df["word_count_tier"] = (
    df["word_count_tier"]
    .astype("object")
    .fillna("NA")
)

bins = [-np.inf, 8000, 15000, 25000, np.inf]
labels = ["<8000", "8000-15000", "15000-25000", "25000+"]

df["char_count_tier"] = pd.cut(
    df["char_count"],
    bins=bins,
    labels=labels,
    right=False
)

df["char_count_tier"] = (
    df["char_count_tier"]
    .astype("object")
    .fillna("NA")
)

df = df.drop(columns = ["word_count", "char_count"])

# The three columns are NULL whenever there is no keyword data.
# We replace those NAs with zero, while also creating a new column which indicates the absence of keyword data
df["no_keyword_data"] = df["competition"].isna().astype(int)
df["search_volume"] = df["search_volume"].fillna(0)
df["competition"] = df["competition"].fillna(0)
df["cpc"] = df["cpc"].fillna(0)
df["keyword_age_days_at_decision"] = df["keyword_age_days_at_decision"].fillna(-30)

# Main intent is NULL whenever it is unknown. NAs in a categorical column can be given a distinct category
df["main_intent"] = df["main_intent"].fillna("NA")

# We create a column to indicate the absence of information about backlinks, and
# replace NAs in backlinks with zeros
df["backlinks_na"] = df["backlinks"].isna().astype(int)
df["backlinks"] = df["backlinks"].fillna(0)



print("The missing values from modifies features:")
print(df.isna().sum())



The missing values from modifies features:
prev_30_impressions                   0
prev_30_avg_position                  0
content_age_days_at_decision          0
days_since_last_update_at_decision    0
keyword_char_count                    0
keyword_token_count                   0
content_type                          0
search_volume                         0
competition                           0
cpc                                   0
keyword_age_days_at_decision          0
main_intent                           0
backlinks                             0
category_count                        0
word_count_tier                       0
char_count_tier                       0
no_keyword_data                       0
backlinks_na                          0
dtype: int64


/tmp/ipykernel_1756/1000416534.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["word_count_tier"] = pd.cut(
/tmp/ipykernel_1756/1000416534.py:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["word_count_tier"] = (
/tmp/ipykernel_1756/1000416534.py:47: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/index

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Two separate tests below, one per attack vector:
1. **Label-derived**: same trap as the main notebook: add the exact quantity that defines the label back in as a feature, watch the AUC jump, remove it.
2. **Future windows**: a per-feature single-column AUC scan (a smell test for anything suspiciously strong on its own), plus a direct check on why guard (c) (`days_since_last_update_at_decision > 0`) exists at all.
3. **Product flags**: We have to make sure `keyword_age_days_at_decision` is always >30, i.e., the keyword was created before the 60-day window we are looking.

In [68]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one -- typing sentences here breaks Run All.

from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
import numpy as np

# ============================================================
# Test 1 -- label-derived columns
# ============================================================
# Rebuild the label the same way the main data-contract notebook does: >20% decline,
# last_30 vs prev_30 gsc_impressions. Pulled back from `features` on purpose -- it's not
# in `df` (section 4's call), this test is exactly why it shouldn't be.
label_df = features.loc[df.index].copy()
label_df["impressions_pct_change"] = (
    (label_df["last_30_impressions"] - label_df["prev_30_impressions"]) / label_df["prev_30_impressions"]
)
is_declining = (label_df["impressions_pct_change"] < -0.20).astype(int)

print(f"is_declining rate: {is_declining.mean():.3f} ({is_declining.sum():,} / {len(is_declining):,})")

# One-hot encode ONCE so the honest and leaked matrices share identical columns/split
X_base = pd.get_dummies(df, columns=["content_type", "main_intent", "word_count_tier", "char_count_tier"])
y = is_declining
groups = features.loc[df.index, "client_hash_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X_base, y, groups))

def client_grouped_auc(X):
    Xtr, Xte = X.iloc[train_idx], X.iloc[test_idx]
    ytr, yte = y.iloc[train_idx], y.iloc[test_idx]
    pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    pipe.fit(Xtr, ytr)
    proba = pipe.predict_proba(Xte)[:, 1]
    return roc_auc_score(yte, proba)

honest_auc = client_grouped_auc(X_base)
print(f"\nTest 1 -- honest {X_base.shape[1]}-column feature matrix AUC: {honest_auc:.3f}")

X_leaked = X_base.copy()
X_leaked["impressions_pct_change"] = label_df["impressions_pct_change"].values
leaked_auc = client_grouped_auc(X_leaked)
print(f"Test 1 -- same matrix + impressions_pct_change added: {leaked_auc:.3f}  <- jumps toward 1.0")
print("Deleting impressions_pct_change now. Keeping the honest number:", round(honest_auc, 3))

# ============================================================
# Test 2 -- future windows
# ============================================================
# 2a. Per-feature smell test: fit nothing, just rank the whole dataset by ONE raw column
# and score it against the label directly. Not a held-out evaluation -- a red-flag scan.
# Anything near 1.0 (in either direction) means "go look at this feature again."
print("\nTest 2a -- single-feature AUC, one column at a time (numeric features only):")
numeric_feats = df.select_dtypes(include=[np.number]).columns
for col in numeric_feats:
    vals = df[col]
    auc = roc_auc_score(y, vals)
    auc = max(auc, 1 - auc)  # a perfect NEGATIVE predictor is just as suspicious as a positive one
    flag = "  <-- SUSPICIOUSLY HIGH, investigate" if auc > 0.85 else ""
    print(f"  {col:38s} single-feature AUC: {auc:.3f}{flag}")
print("We see no strange signs!")

# 2b. Why guard (c) (days_since_last_update_at_decision > 0) exists: a direct count,
# across all of dim_content, of rows that would have failed it -- i.e. content updated
# AFTER the decision moment, which would have made this feature encode future information.
future_updates = con.sql(f"""
    SELECT COUNT(*) AS rows_updated_after_decision_moment
    FROM {DIM_CONTENT}
    WHERE content_updated_date > DATE '2026-05-01'
""").df()
print(f"\nTest 2b -- dim_content rows with content_updated_date after the 2026-05-01 decision moment:")
print(future_updates)
print("-> guard (c) exists to drop exactly these from the feature frame.")

# ============================================================
# Test 3 -- product flags
# ============================================================
# Make sure the keyword was created before the 60-day window

df_keyword_available = df[df["keyword_age_days_at_decision"] != -30] # all the missing values were given the value -30

print("The minimum value of keyword_age_days_at_decision:")
print(df_keyword_available["keyword_age_days_at_decision"].min())

print(""" We can see that keyword_age_days_at_decision is always greater than 30,
meaning all keywords were created before our time window.
""")

is_declining rate: 0.543 (10,279 / 18,918)

Test 1 -- honest 32-column feature matrix AUC: 0.581
Test 1 -- same matrix + impressions_pct_change added: 1.000  <- jumps toward 1.0
Deleting impressions_pct_change now. Keeping the honest number: 0.581

Test 2a -- single-feature AUC, one column at a time (numeric features only):
  prev_30_impressions                    single-feature AUC: 0.550
  prev_30_avg_position                   single-feature AUC: 0.540
  content_age_days_at_decision           single-feature AUC: 0.588
  days_since_last_update_at_decision     single-feature AUC: 0.514
  keyword_char_count                     single-feature AUC: 0.543
  keyword_token_count                    single-feature AUC: 0.537
  search_volume                          single-feature AUC: 0.582
  competition                            single-feature AUC: 0.548
  cpc                                    single-feature AUC: 0.528
  keyword_age_days_at_decision           single-feature AUC: 0.581
  ba

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

`client_hash_id, content_hash_id, client_hash_id_1, keyword_hash_id, url_hash_id`: just context

`last_30_impressions`: a proxy

`first_seen`: Not relevant

`url_char_count`: Not relevant

`content_created_date, content_updated_date, keyword_created_date`: Already using their difference from decision date.

`competition_level`: just categorical version of `competition`

`provider_used, model_used`: Not relevant

`char_count, word_count`: already using their binned verions

`last_optimized_date, optimization_eligible_date`: Overwhelming majority is NA.


`is_published`: Almost everywhere is True

`is_deleted`: Almost everywhere is False

In [64]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print(f"""Number of rows with is_published = False:
    {len(
        features[features["is_published"] == False]
    )}
""")

print(f"""Number of rows with is_deleted = True:
    {len(
        features[features["is_deleted"] == True]
    )}
""")

Number of rows with is_published = False:
    9

Number of rows with is_deleted = True:
    8



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.